<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "270"
iso_country = "Gambia"
year = 2020
import_portfolio = "h4_ag6_by_sea_M.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h4_ag6_by_sea_DX.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 1.810*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Gambia': 

Number of trade records present in dataset: 12429
Number of partner countries present in dataset: 110
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$524.2m
Total weight within the dataset: 1006.3 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,270,Gambia,036,Australia,Import,Sea,630130,630130 | Blankets (other than,6301,6301 | Blankets and travell,63,"63 | Textiles, made up ar",2711,Blankets and travelling rugs (except electric ...,Textiles and Wearing Apparel,27,TEX - Textiles,10.070,0,10.070,kg,70.0,70.0,1.830726,18631.0,11576.763101,21.193881,['Container'],['Container'],0.097054,0.096869,0.096888,0.096888,0.096971,0.096971,0.101633,0.181492,0.111114,0.104230,0.104531,0.106946,0.099467,0.177174,0.117554,0.108055,0.105466,0.110396,0.097568,0.194458,0.146227,0.128019,0.118267,0.115136,0.095703,0.206205,0.169512,0.157681,0.126492,0.123490,0.093874,0.210252,0.202174,0.204699,0.136122,0.129760,0.092008,0.205835,0.195292,0.213015,0.161267,0.132178,0.090250,0.182192,0.191053,0.199782,0.146734,0.133743,0.088457,0.167496,0.169811,0.204845,0.170747,0.134671,2711
1,1,270,Gambia,699,India,Import,Sea,630130,630130 | Blankets (other than,6301,6301 | Blankets and travell,63,"63 | Textiles, made up ar",2711,Blankets and travelling rugs (except electric ...,Textiles and Wearing Apparel,27,TEX - Textiles,25.185,0,25.185,kg,35.0,35.0,4.578633,12880.0,8003.258480,36.643983,['Container'],['Container'],0.167806,0.167486,0.167519,0.167519,0.167661,0.167661,0.175723,0.313797,0.192115,0.180213,0.180732,0.184908,0.171978,0.306331,0.203250,0.186825,0.182349,0.190874,0.168693,0.336215,0.252825,0.221343,0.204482,0.199069,0.165470,0.356525,0.293085,0.272628,0.218703,0.213512,0.162307,0.363524,0.349556,0.353922,0.235354,0.224354,0.159080,0.355886,0.337657,0.368300,0.278829,0.228534,0.156041,0.315007,0.330328,0.345421,0.253701,0.231239,0.152942,0.289599,0.293601,0.354175,0.295219,0.232845,2711


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $153.0m (29.2%), ... 

                             cifvalue  vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                           
10 | Cereals               44627807.4  77825970.5      8.5      7.7
17 | Sugars and sugar con  26517881.0  56642973.3      5.1      5.6
15 | Animal, vegetable or  23239924.0  11744519.0      4.4      1.2
11 | Products of the mill  19940202.7  48261202.5      3.8      4.8
02 | Meat and edible meat   8369923.9   4481084.3      1.6      0.4
04 | Dairy produce; birds   7746756.0   4267711.4      1.5      0.4
19 | Preparations of cere   5154144.5   2567812.5      1.0      0.3
21 | Miscellaneous edible   4306984.3   1776515.6      0.8      0.2
22 | Beverages, spirits a   3323265.7   5946902.1      0.6      0.6
20 | Preparations of vege   2886831.3   2561465.7      0.6      0.3
07 | Vegetables and certa   2502143.9   4002858.8      0.5      0.4
09 | Coffee, te

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $154.3m (29.4%), ... 

                     cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Agriculture       109130345.8  151456286.8     20.8     15.1
Food & Beverages   45067539.8   70035863.4      8.6      7.0
Fishing               89707.6      27463.4      0.0      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $149.8m (28.6%), ... 

                                 cifvalue  vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                              
PCR - Processed rice           44344557.7  77026496.8      8.5      7.7
OFD - Food products n.e.c.     37358430.1  56882995.0      7.1      5.7
SGR - Sugar                    25576691.5  56330140.6      4.9      5.6
VOL - Vegetable oils and fats  23242465.5  11760910.7      4.4      1.2
OMT - Meat products n.e.c.      7915511.1   4330425.6      1.5      0.4
MIL - Dairy products            7703319.0   4247986.1      1.5      0.4
V_F - Vegetables, fruit, nuts   2320301.2   3834293.7      0.4      0.4
CMT - Bovine meat prods          473318.6    160204.6      0.1      0.0
OCR - Crops n.e.c.               465899.4    140455.7      0.1      0.0
GRO - Cereal grains n.e.c.       266123.1    769724.9      0.1      0.1
PFB - Plant-based fibers          51870.0     92920.7      0.0      0.

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 29.2% 
GTAP 28.6% 
EORA 29.4%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_46372/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,270,Gambia,036,Australia,Import,Sea,630130,630130 | Blankets (other than,6301,6301 | Blankets and travell,63,"63 | Textiles, made up ar",2711,Blankets and travelling rugs (except electric ...,Textiles and Wearing Apparel,27,TEX - Textiles,10.070,0,10.070,kg,70.0,70.0,1.830726,18631.0,11576.763101,21.193881,['Container'],['Container'],0.097054,0.096869,0.096888,0.096888,0.096971,0.096971,0.101633,0.181492,0.111114,0.104230,0.104531,0.106946,0.099467,0.177174,0.117554,0.108055,0.105466,0.110396,0.097568,0.194458,0.146227,0.128019,0.118267,0.115136,0.095703,0.206205,0.169512,0.157681,0.126492,0.123490,0.093874,0.210252,0.202174,0.204699,0.136122,0.129760,0.092008,0.205835,0.195292,0.213015,0.161267,0.132178,0.090250,0.182192,0.191053,0.199782,0.146734,0.133743,0.088457,0.167496,0.169811,0.204845,0.170747,0.134671,2711,NaN
1,1,270,Gambia,699,India,Import,Sea,630130,630130 | Blankets (other than,6301,6301 | Blankets and travell,63,"63 | Textiles, made up ar",2711,Blankets and travelling rugs (except electric ...,Textiles and Wearing Apparel,27,TEX - Textiles,25.185,0,25.185,kg,35.0,35.0,4.578633,12880.0,8003.258480,36.643983,['Container'],['Container'],0.167806,0.167486,0.167519,0.167519,0.167661,0.167661,0.175723,0.313797,0.192115,0.180213,0.180732,0.184908,0.171978,0.306331,0.203250,0.186825,0.182349,0.190874,0.168693,0.336215,0.252825,0.221343,0.204482,0.199069,0.165470,0.356525,0.293085,0.272628,0.218703,0.213512,0.162307,0.363524,0.349556,0.353922,0.235354,0.224354,0.159080,0.355886,0.337657,0.368300,0.278829,0.228534,0.156041,0.315007,0.330328,0.345421,0.253701,0.231239,0.152942,0.289599,0.293601,0.354175,0.295219,0.232845,2711,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $140.0m of $524.2m (26.7%)
Volume: 210.8 of 1006.3 million kilograms (20.9%)
Value-to-Weight: 0.66 from 0.52 US$ per kg (+27.5%)
Average distance to market (volume-weighted): 4970.8 from 3371.2 nm (+47.5%)

            food_group  N_com    cifvalue   vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    530  70845511.6  128266290.9     13.5     12.7
1      Sugar and Honey    103  26600677.0   56666710.9      5.1      5.6
2        Fats and Oils    110  23231068.3   11736224.6      4.4      1.2
3        Meat and Prep     52   8018122.7    4344795.3      1.5      0.4
4                Dairy    185   7503622.8    4181897.5      1.4      0.4
5           Vegetables    161   3307310.9    4661836.0      0.6      0.5
6  Unprocessed Cereals     28    266123.1     769724.9      0.1      0.1
7               Fruits     49    240573.6     193434.2      0.0      0.0



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,12551192.2,22402994.3,13720317.3,12870656.2,12907740.1,13206427.3,11593913.6,25952485.2,24955359.2,25269092.6,16807038.9,16020772.5,10925708.6,20678150.0,20964028.3,25285137.4,21076796.5,16626519.3
1,Fats and Oils,52765627.3,94217724.4,57686451.7,54112532.7,54268635.7,55522993.5,48737810.9,109146057.0,104952864.0,106265528.0,70668245.2,67365167.9,45926291.7,86953634.4,88155745.0,106340260.5,88639014.9,69914183.5
2,Fruits,479614.7,856440.8,524350.2,491864.7,493283.6,504681.8,442999.3,992159.7,954036.9,965958.1,642359.6,612335.6,417440.9,790406.1,801329.3,966643.5,805737.3,635508.1
3,Meat and Prep,9455452.5,16885048.7,10337496.9,9697022.9,9724999.6,9949688.4,8733535.2,19560804.4,18809205.0,19044122.0,12664100.5,12072228.0,8229618.4,15582977.6,15798330.6,19057758.1,15885386.5,12529108.4
4,Processed Cereals,268379175.9,424621689.4,283712145.8,269221002.5,269920311.3,278110231.4,253200388.8,510555166.0,486149471.0,500926880.2,350934204.3,321382514.8,242605206.1,414339064.7,417989554.4,482793773.5,409434277.1,330163029.9
5,Sugar and Honey,97228409.6,168708514.4,105427683.2,99166529.9,99445386.8,101923211.5,90282952.2,197008879.7,189036878.4,192192858.3,129415166.5,122222744.0,85434844.3,157676417.4,159684537.5,190700069.8,159568441.7,126545350.8
6,Unprocessed Cereals,2900157.6,4962357.3,3133646.4,2950358.2,2958048.3,3034727.7,2699803.3,5778503.4,5546994.3,5655127.1,3831070.0,3619307.5,2559950.8,4654033.3,4714840.9,5601948.8,4690178.0,3743667.6
7,Vegetables,10969495.0,19381457.2,11955744.4,11224067.6,11255650.6,11525355.7,10152935.8,22463878.8,21587728.9,21900204.4,14626252.8,13925176.0,9581061.4,17945860.2,18199958.1,21860082.8,18227520.1,14440256.0


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,1.31,0.16,0.04,0.05,0.09,1.91,1.78,1.82,0.69,0.59,1.30,1.34,1.91,1.35,0.76
1,Fats and Oils,1.78,0.21,0.06,0.06,0.12,2.60,2.42,2.48,0.94,0.80,1.77,1.82,2.60,1.84,1.03
2,Fruits,1.57,0.19,0.05,0.06,0.10,2.28,2.12,2.17,0.83,0.70,1.55,1.60,2.28,1.61,0.91
3,Meat and Prep,0.93,0.11,0.03,0.03,0.06,1.35,1.26,1.29,0.49,0.42,0.92,0.94,1.35,0.95,0.54
4,Processed Cereals,2.21,0.22,0.01,0.02,0.14,3.63,3.29,3.50,1.38,0.96,2.42,2.48,3.39,2.35,1.24
5,Sugar and Honey,2.69,0.31,0.07,0.08,0.18,4.01,3.71,3.83,1.47,1.20,2.72,2.79,3.96,2.79,1.55
6,Unprocessed Cereals,7.75,0.88,0.19,0.22,0.51,11.57,10.70,11.11,4.25,3.46,7.87,8.10,11.43,8.00,4.45
7,Vegetables,2.54,0.30,0.08,0.09,0.17,3.72,3.46,3.55,1.35,1.14,2.53,2.61,3.71,2.61,1.47
